# Minimal GPT Attention Math Walkthrough

Inspired by Karpathy's `minGPT`, this notebook rebuilds the scaled dot-product attention equations step-by-step so you can relate them to linear index models and conditional expectations.


## Outline

1. Build a toy causal self-attention module from scratch.
2. Show how scaling by `1/√d` mirrors variance stabilisation.
3. Connect the resulting matrix multiplications to econometric estimators.


In [ ]:
import torch
import torch.nn as nn

class TinyAttention(nn.Module):
    def __init__(self, d_model: int = 12, n_heads: int = 3):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x: torch.Tensor):
        B, T, C = x.shape
        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)

        def reshape_heads(t):
            return t.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        q, k, v = map(reshape_heads, (q, k, v))
        scores = (q @ k.transpose(-2, -1)) / torch.sqrt(torch.tensor(self.head_dim))
        attn = scores.softmax(dim=-1)
        context = attn @ v
        context = context.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(context), attn


In [ ]:
torch.manual_seed(0)
module = TinyAttention()
inputs = torch.randn(1, 5, module.d_model)
outputs, attn = module(inputs)
attn.shape

### Variance stabilisation view

Dividing by `√d` keeps the variance of the dot product roughly constant as dimensionality grows. This mirrors the econometric practice of rescaling regressors to stabilise the information matrix (`X'X/n`).


In [ ]:
head = 0
weights = attn[0, head]
print("Attention weights for head 0:\n", weights)

value_vectors = module.qkv_proj(inputs).chunk(3, dim=-1)[2]
value_vectors = value_vectors.view(1, 5, module.n_heads, module.head_dim)[:, :, head]
conditional_expectation = weights @ value_vectors
conditional_expectation

### Linear index interpretation

Each score `s_{ij} = q_i^\top k_j` acts like a linear index `β^Tx`. Identification arguments familiar from binary response models apply: altering `k_j` or `q_i` shifts the index, leading to predictable changes in probabilities after the softmax transformation.


## Econometric takeaways

- The attention mechanism is a matrix of **weights** derived from a bilinear index. This keeps it squarely within the econometric mental model of estimators.
- Conditional expectation analogies (`E[V | Q]`) help audiences anchor the meaning of value aggregation.
- The module above is small enough to paste into lectures or collaborate on derivations live.
